# Giai đoạn 1: Khám phá dữ liệu (Exploratory Data Analysis)
**Dự án:** Phân tích Dữ liệu Thương mại Điện tử Olist (Brazil, 2016–2018)

### Mục tiêu cốt lõi:
1. **Nắm quy mô dữ liệu:** Ghi tổng số dòng và số cột của từng bảng trong 9 bảng dữ liệu gốc, làm mốc "trước khi làm sạch".
2. **Xác định kiểu dữ liệu của từng thuộc tính:** Với mỗi cột, ghi kiểu đang lưu và phân loại theo ý nghĩa (Numerical, Categorical, Datetime, Boolean, định danh); chỉ ra các cột lưu sai kiểu.
3. **Mô tả đặc điểm các biến bằng thống kê mô tả:**
   - Biến số: mean, median, standard deviation, max, min (bổ sung variance).
   - Biến phân loại: số giá trị khác nhau, mode, tần suất (bổ sung mode cho biến số rời rạc như `review_score`).
4. **Ghi nhận quan sát định tính ban đầu:** Nêu các đặc điểm và quy luật nhìn thấy trong dữ liệu, mỗi quan sát kèm con số cụ thể.

## Cấu hình Môi trường & Nạp 9 Bảng Dữ Liệu Gốc
- Thiết lập đường dẫn linh hoạt tự động tìm thư mục `data/raw/`.
- Đọc mã bưu chính (`zip_code_prefix`) ở dạng chuỗi ký tự (`dtype=str`) để bảo toàn các số 0 ở đầu.

In [1]:
import os
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

# 1. Cấu hình hiển thị và cảnh báo
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# 2. Xác định thư mục gốc dự án linh hoạt (chứa thư mục data/raw)
current_dir = Path.cwd()
if (current_dir / 'data' / 'raw').exists():
    BASE_DIR = current_dir
elif (current_dir.parent / 'data' / 'raw').exists():
    BASE_DIR = current_dir.parent
elif (current_dir.parents[1] / 'data' / 'raw').exists():
    BASE_DIR = current_dir.parents[1]
else:
    BASE_DIR = Path('c:/ADY291c-AI2107-Group2')

RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
print(f"Thư mục dữ liệu gốc (Raw Data): {RAW_DATA_DIR}")

# 3. Bản đồ 9 file dữ liệu thô
raw_files_map = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

# 4. Đọc dữ liệu, giữ nguyên mã zip code ở dạng chuỗi ký tự
zip_dtype = {
    'customer_zip_code_prefix': str,
    'seller_zip_code_prefix': str,
    'geolocation_zip_code_prefix': str
}

dfs: Dict[str, pd.DataFrame] = {}
for name, filename in raw_files_map.items():
    fpath = RAW_DATA_DIR / filename
    if not fpath.is_file():
        raise FileNotFoundError(f"Không tìm thấy file: {filename} tại {RAW_DATA_DIR}")
    dfs[name] = pd.read_csv(fpath, dtype=zip_dtype)

print("Đã nạp thành công 9 bảng dữ liệu Olist vào bộ nhớ.")

Thư mục dữ liệu gốc (Raw Data): C:\ADY291c-AI2107-Group2\data\raw


Đã nạp thành công 9 bảng dữ liệu Olist vào bộ nhớ.


## 1: Quy Mô Dữ Liệu (Số Dòng, Số Cột Từng Bảng)
Sử dụng thuộc tính `df.shape` để tổng hợp số dòng và số cột của cả 9 bảng dữ liệu thô vào một bảng tóm tắt 9 dòng duy nhất.

In [2]:
# Bước 1: Sử dụng df.shape để tóm tắt quy mô 9 bảng dữ liệu
table_shapes = []
for name, df in dfs.items():
    n_rows, n_cols = df.shape
    table_shapes.append({
        'Tên Bảng Dữ Liệu': f"olist_{name}_dataset" if name != 'category_translation' else 'product_category_name_translation',
        'Số Dòng (Rows)': f"{n_rows:,}",
        'Số Cột (Columns)': n_cols
    })

df_shapes_summary = pd.DataFrame(table_shapes)
print("--- BẢNG TÓM TẮT QUY MÔ 9 BẢNG DỮ LIỆU GỐC ---")
display(df_shapes_summary)

--- BẢNG TÓM TẮT QUY MÔ 9 BẢNG DỮ LIỆU GỐC ---


,Tên Bảng Dữ Liệu,Số Dòng (Rows),Số Cột (Columns)
0,olist_orders_dataset,"99,441",8
1,olist_order_items_dataset,"112,650",7
2,olist_payments_dataset,"103,886",5
3,olist_reviews_dataset,"99,224",7
4,olist_products_dataset,"32,951",9
5,olist_customers_dataset,"99,441",5
6,olist_sellers_dataset,"3,095",4
7,olist_geolocation_dataset,"1,000,163",5
8,product_category_name_translation,71,2


## 2: Kiểu Dữ Liệu của Từng Thuộc Tính (Data Profiling & Classification)
Khảo sát toàn bộ các thuộc tính bằng `df.dtypes` / `df.info()`, phân loại từng cột thành:
- **ID Key (Định danh):** Các cột khóa chính, khóa ngoại hoặc mã bưu chính (`order_id`, `customer_id`, `product_id`, `seller_id`, mã zip).
- **Numerical:** Biến số thực / số nguyên đo lường định lượng (`price`, `freight_value`, `payment_value`, kích thước, khối lượng, điểm đánh giá).
- **Categorical:** Biến phân loại / chuỗi văn bản (`order_status`, `payment_type`, bang, thành phố, tên danh mục).
- **Datetime:** Các mốc thời gian diễn ra vòng đời đơn hàng và đánh giá.
- **Boolean:** Các biến cờ nhị phân (nếu có).

> **Ghi chú kỹ thuật:** Chỉ rõ các cột đang bị lưu **sai kiểu** trong dữ liệu gốc (đặc biệt là 8 cột mốc thời gian đang bị lưu dạng chuỗi `object` cần được ép về `datetime64[ns]` ở Giai đoạn 2 để phục vụ tính toán thời gian giao vận và trễ hạn).

In [3]:
# Bước 2: Kiểm tra df.dtypes và phân loại ngữ nghĩa cho toàn bộ 52 thuộc tính
columns_profile = []

for table_name, df in dfs.items():
    for col, dtype in df.dtypes.items():
        dtype_str = str(dtype)
        
        # Phân loại nhóm kiểu ngữ nghĩa (Semantic Type)
        if any(t_key in col.lower() for t_key in ['date', 'timestamp']) or col.lower().endswith('_at'):
            semantic_group = 'Datetime'
            tech_note = 'ĐANG SAI KIỂU: Lưu dạng text (object), cần ép sang datetime64[ns] ở Giai đoạn 2'
        elif col.endswith('_id') or 'zip_code' in col.lower() or col in ['order_item_id', 'payment_sequential']:
            semantic_group = 'ID Key (Định danh)'
            tech_note = 'Khóa định danh / Mã bưu chính (cần giữ dạng chuỗi zfill(5))'
        elif pd.api.types.is_numeric_dtype(dtype):
            if col == 'review_score':
                semantic_group = 'Numerical (Ordinal)'
                tech_note = 'Biến thứ bậc / Điểm đánh giá (1 đến 5 sao)'
            else:
                semantic_group = 'Numerical (Số học)'
                tech_note = 'Chuẩn định lượng số thực/nguyên'
        elif pd.api.types.is_bool_dtype(dtype):
            semantic_group = 'Boolean'
            tech_note = 'Biến nhị phân Đúng/Sai'
        else:
            semantic_group = 'Categorical (Phân loại)'
            tech_note = 'Chuẩn biến định tính / Chuỗi văn bản'
            
        columns_profile.append({
            'Tên Bảng': table_name,
            'Tên Cột (Thuộc tính)': col,
            'Kiểu Lưu Gốc (Raw Dtype)': dtype_str,
            'Nhóm Kiểu Ngữ Nghĩa': semantic_group,
            'Ghi Chú Đánh Giá Kỹ Thuật': tech_note
        })

df_columns_profile = pd.DataFrame(columns_profile)

# 1. Thống kê tóm tắt số lượng cột theo từng nhóm kiểu ngữ nghĩa
print("--- THỐNG KÊ PHÂN BỐ CÁC NHÓM KIỂU DỮ LIỆU ---")
display(df_columns_profile['Nhóm Kiểu Ngữ Nghĩa'].value_counts().to_frame(name='Số Lượng Thuộc Tính'))

# 2. Hiển thị bảng chi tiết Data Dictionary của toàn bộ 52 thuộc tính
print(f"\n--- BẢNG DATA DICTIONARY CHI TIẾT ({len(df_columns_profile)} THUỘC TÍNH) ---")
display(df_columns_profile)

--- THỐNG KÊ PHÂN BỐ CÁC NHÓM KIỂU DỮ LIỆU ---


,Số Lượng Thuộc Tính
Nhóm Kiểu Ngữ Nghĩa,
ID Key (Định danh),17
Categorical (Phân loại),13
Numerical (Số học),13
Datetime,8
Numerical (Ordinal),1



--- BẢNG DATA DICTIONARY CHI TIẾT (52 THUỘC TÍNH) ---


,Tên Bảng,Tên Cột (Thuộc tính),Kiểu Lưu Gốc (Raw Dtype),Nhóm Kiểu Ngữ Nghĩa,Ghi Chú Đánh Giá Kỹ Thuật
0,orders,order_id,str,ID Key (Định danh),Khóa định danh / Mã bưu chính (cần giữ dạng ch...
1,orders,customer_id,str,ID Key (Định danh),Khóa định danh / Mã bưu chính (cần giữ dạng ch...
2,orders,order_status,str,Categorical (Phân loại),Chuẩn biến định tính / Chuỗi văn bản
3,orders,order_purchase_timestamp,str,Datetime,"ĐANG SAI KIỂU: Lưu dạng text (object), cần ép ..."
4,orders,order_approved_at,str,Datetime,"ĐANG SAI KIỂU: Lưu dạng text (object), cần ép ..."
5,orders,order_delivered_carrier_date,str,Datetime,"ĐANG SAI KIỂU: Lưu dạng text (object), cần ép ..."
6,orders,order_delivered_customer_date,str,Datetime,"ĐANG SAI KIỂU: Lưu dạng text (object), cần ép ..."
7,orders,order_estimated_delivery_date,str,Datetime,"ĐANG SAI KIỂU: Lưu dạng text (object), cần ép ..."
8,order_items,order_id,str,ID Key (Định danh),Khóa định danh / Mã bưu chính (cần giữ dạng ch...
9,order_items,order_item_id,int64,ID Key (Định danh),Khóa định danh / Mã bưu chính (cần giữ dạng ch...


## 3: Thống Kê Mô Tả (Descriptive Statistics - Chuẩn CLO6)
Trình bày kết quả thống kê mô tả hai tầng bằng hai bảng tổng hợp tinh gọn, không dán nguyên bảng đầu ra thô dài dòng:
1. **Biến định lượng (Numerical Variables):** Sử dụng `df.describe()` kết hợp tính toán thêm **Variance (Phương sai)** và **Mode (Yếu vị)** theo yêu cầu chuẩn CLO6 cho 5 thuộc tính chủ chốt gắn với mục tiêu dự đoán Review Score: `price`, `freight_value`, `payment_value`, `payment_installments`, `review_score`.
2. **Biến định tính (Categorical Variables):** Sử dụng `df.describe(include='object')` và `value_counts()` khảo sát số lượng giá trị phân biệt (`Unique`), giá trị xuất hiện nhiều nhất (`Mode`), tần suất (`Frequency`) và tỷ lệ % của mode cho 3 thuộc tính: `order_status`, `payment_type`, `customer_state`.

In [4]:
# Bước 3.1: Thống kê mô tả Biến Số Học (bổ sung Variance và Mode theo chuẩn CLO6)
numeric_vars = {
    'price': dfs['order_items']['price'],
    'freight_value': dfs['order_items']['freight_value'],
    'payment_value': dfs['payments']['payment_value'],
    'payment_installments': dfs['payments']['payment_installments'],
    'review_score': dfs['reviews']['review_score']
}

num_summary_records = []
for var_name, series in numeric_vars.items():
    s_valid = series.dropna()
    mode_val = s_valid.mode()[0] if not s_valid.empty else np.nan
    
    num_summary_records.append({
        'Tên Biến': var_name,
        'Số Lượng (Count)': f"{int(s_valid.count()):,}",
        'Trung Bình (Mean)': f"{s_valid.mean():,.2f}",
        'Trung Vị (Median)': f"{s_valid.median():,.2f}",
        'Độ Lệch Chuẩn (Std)': f"{s_valid.std():,.2f}",
        'Phương Sai (Variance)': f"{s_valid.var():,.2f}",
        'Yếu Vị (Mode)': f"{mode_val:,.2f}",
        'Tối Thiểu (Min)': f"{s_valid.min():,.2f}",
        'Tối Đa (Max)': f"{s_valid.max():,.2f}"
    })

df_num_summary = pd.DataFrame(num_summary_records)
print("--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH LƯỢNG (NUMERICAL - CLO6) ---")
display(df_num_summary)

--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH LƯỢNG (NUMERICAL - CLO6) ---


,Tên Biến,Số Lượng (Count),Trung Bình (Mean),Trung Vị (Median),Độ Lệch Chuẩn (Std),Phương Sai (Variance),Yếu Vị (Mode),Tối Thiểu (Min),Tối Đa (Max)
0,price,"112,650",120.65,74.99,183.63,"33,721.42",59.90,0.85,"6,735.00"
1,freight_value,"112,650",19.99,16.26,15.81,249.84,15.10,0.00,409.68
2,payment_value,"103,886",154.10,100.00,217.49,"47,303.67",50.00,0.00,"13,664.08"
3,payment_installments,"103,886",2.85,1.00,2.69,7.22,1.00,0.00,24.00
4,review_score,"99,224",4.09,5.00,1.35,1.82,5.00,1.00,5.00


In [5]:
# Bước 3.2: Thống kê mô tả Biến Phân Loại (Categorical Variables)
key_cat_targets = [
    ('orders', 'order_status', 'Trạng thái đơn hàng'),
    ('payments', 'payment_type', 'Phương thức thanh toán'),
    ('customers', 'customer_state', 'Bang của khách hàng')
]

cat_summary_records = []
for tbl_name, col_name, desc in key_cat_targets:
    s_valid = dfs[tbl_name][col_name].dropna()
    val_counts = s_valid.value_counts()
    top_mode = val_counts.index[0] if len(val_counts) > 0 else 'N/A'
    mode_freq = val_counts.iloc[0] if len(val_counts) > 0 else 0
    mode_pct = (mode_freq / len(s_valid) * 100) if len(s_valid) > 0 else 0.0
    
    cat_summary_records.append({
        'Bảng Dữ Liệu': tbl_name,
        'Tên Biến': col_name,
        'Mô Tả Thuộc Tính': desc,
        'Số Giá Trị Khác Nhau (Unique)': f"{s_valid.nunique():,}",
        'Mode (Phổ biến nhất)': top_mode,
        'Tần Suất Mode (Frequency)': f"{mode_freq:,}",
        'Tỷ Lệ Mode (%)': f"{mode_pct:.2f}%"
    })

df_cat_summary = pd.DataFrame(cat_summary_records)
print("--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH TÍNH (CATEGORICAL) ---")
display(df_cat_summary)

--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH TÍNH (CATEGORICAL) ---


,Bảng Dữ Liệu,Tên Biến,Mô Tả Thuộc Tính,Số Giá Trị Khác Nhau (Unique),Mode (Phổ biến nhất),Tần Suất Mode (Frequency),Tỷ Lệ Mode (%)
0,orders,order_status,Trạng thái đơn hàng,8,delivered,"96,478",97.02%
1,payments,payment_type,Phương thức thanh toán,5,credit_card,"76,795",73.92%
2,customers,customer_state,Bang của khách hàng,27,SP,"41,746",41.98%


## Bước 4: Nhận Xét Định Tính Ban Đầu (Initial Observations)
Dựa trên các bảng số liệu thống kê mô tả thu thập được ở trên, dưới đây là **8 quan sát định tính ban đầu kèm số liệu thực tế chứng minh** (thực hiện theo nguyên tắc mô tả khách quan các đặc điểm của tập dữ liệu, không vội vàng giải thích nguyên nhân nhân quả khi chưa có kiểm định chuyên sâu):

1. **Trạng thái hoàn tất đơn hàng:** Trong **8 trạng thái** đơn hàng ghi nhận được, đại đa số ở trạng thái giao thành công (`delivered`) với **96,478 trên tổng số 99,441 đơn**, chiếm **97.02%**; các trạng thái còn lại (`shipped`, `canceled`, `unavailable`, `invoiced`...) chỉ chiếm **2.98%**.
2. **Điểm đánh giá khách hàng (review_score):** Phân bố điểm số tập trung mạnh ở nhóm hài lòng: mức 5 sao xuất hiện nhiều nhất (**Mode = 5.0**, Trung vị **Median = 5.0**) với **57,328 lượt** (chiếm **57.78%**), trong khi **11,424 lượt** là 1 sao (**11.51%**). Điểm trung bình đạt **4.09** với độ lệch chuẩn **1.35**, cho thấy đa số đánh giá tích cực nhưng vẫn tồn tại nhóm điểm thấp kéo lệch phân bố; đây là biến mục tiêu của dự án.
3. **Phân phối lệch phải của giá bán (price):** Giá sản phẩm có trung vị **74.99 BRL** thấp hơn nhiều so với trung bình **120.65 BRL**; độ lệch chuẩn **183.63 BRL** cùng phương sai **33,721.42** rất lớn, giá trị trải từ **0.85 BRL** đến **6,735.00 BRL**, phản ánh đặc trưng đuôi dài (right-skewed) với một số đơn hàng giá trị cao.
4. **Chi phí vận chuyển (freight_value):** Phí vận chuyển trung bình là **19.99 BRL**, trung vị **16.26 BRL**, độ lệch chuẩn **15.81 BRL**; giá trị dao động từ **0.00 BRL** (đơn hàng được miễn phí vận chuyển) đến **409.68 BRL**.
5. **Phương thức thanh toán chủ đạo:** Thẻ tín dụng (`credit_card`) là hình thức thanh toán chiếm ưu thế áp đảo với **76,795 giao dịch** (**73.92%**), tiếp theo là hình thức `boleto` với **19,784 giao dịch** (**19.05%**), `voucher` (**5.56%**), thẻ ghi nợ `debit_card` (**1.47%**) và **3 giao dịch** còn lại mang nhãn `not_defined`.
6. **Kỳ hạn trả góp (payment_installments):** Yếu vị và trung vị đều là **1 kỳ** (**Mode = 1.00**, **Median = 1.00**), nghĩa là phần lớn giao dịch thanh toán một lần; tuy nhiên số kỳ trả góp tối đa lên đến **24 kỳ**, kỳ hạn trung bình toàn hệ thống là **2.85 kỳ** (độ lệch chuẩn **2.69**).
7. **Độ phân tán của giá trị thanh toán (payment_value):** Giá trị thanh toán trung vị là **100.00 BRL**, trung bình là **154.10 BRL**, nhưng mức tối đa đạt tới **13,664.08 BRL** khiến phương sai cực kỳ lớn (**47,303.67**) và độ lệch chuẩn **217.49**, chứng tỏ có sự khác biệt quy mô giỏ hàng rất lớn giữa các đơn hàng phổ thông và đơn sỉ/giá trị cao.
8. **Mật độ tập trung địa lý:** Trong **27 bang/thành phố**, khách hàng tập trung áp đảo tại bang Sao Paulo (`SP`) với **41,746 khách hàng** (chiếm **41.98%**), kế tiếp là Rio de Janeiro (`RJ`, **12,852 khách hàng**, **12.92%**) và Minas Gerais (`MG`, **11,635 khách hàng**, **11.70%**), phản ánh nhu cầu thương mại điện tử phân bố đậm đặc tại khu vực Đông Nam Brazil.